# Root App — Audio Generator (runs entirely in your browser, free GPU)

Works on Android via Chrome — no computer needed. Steps:
1. Run each cell below in order (tap the ▶ button on the left of each cell)
2. Cell 3 will ask you to upload `root-audio-checklist.csv` — export that from the app: Settings → Speech audio → Export
3. Cell 5 generates the audio (this is the slow part — grab a coffee)
4. Cell 6 zips everything and downloads it straight to your phone
5. Unzip on your phone, upload the resulting `audio` folder to GitHub via Codespaces

**Before you start:** go to Runtime → Change runtime type → set Hardware accelerator to **GPU** (free tier is fine). This makes generation dramatically faster.

## 1. Install dependencies (first run only, ~2 minutes)

In [ ]:
!pip install -q TTS pydub
!apt-get -qq install -y ffmpeg
print('Done installing.')

## 2. Load the XTTS-v2 model (downloads ~2GB the first time)

In [ ]:
from TTS.api import TTS
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('WARNING: no GPU detected. Go to Runtime > Change runtime type > GPU, then re-run from the top.')

tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('Model loaded.')

## 3. Upload your checklist CSV
Export it from the app first: **Settings → Speech audio → Export**. Then run this cell and pick the file from your phone's storage/downloads.

In [ ]:
from google.colab import files
import csv

uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]

rows = []
with open(csv_filename, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)
print(f'Loaded {len(rows)} total rows from the checklist.')
print('Languages found:', sorted(set(r['language'] for r in rows)))
print('Dialects found:', sorted(set(r['dialect'] for r in rows)))

## 4. Choose what to generate this run
XTTS-v2 supports German directly. It does **not** support Irish or Vietnamese — running those rows will fail. Leave this cell's filters set to German for now; we'll handle Irish/Vietnamese separately.

For a real Austrian/Swiss accent instead of the same voice reading three times, upload a short (6+ second) clean speech sample of a native speaker and set `speaker_wav_path` below. Otherwise leave it as `None` to use XTTS's default voice.

In [ ]:
LANGUAGE = 'German'
DIALECT = 'Germany'   # change to 'Austria' or 'Switzerland' for the other two runs
XTTS_LANG_CODE = 'de'
speaker_wav_path = None  # e.g. 'austrian_sample.wav' after uploading one — see note above

filtered = [r for r in rows if r['language'] == LANGUAGE and r['dialect'] == DIALECT]
print(f'{len(filtered)} clips match language={LANGUAGE!r} dialect={DIALECT!r}')
for r in filtered[:5]:
    print(' ', r['filename'], '<-', r['text'])
if len(filtered) > 5:
    print(f'  ... and {len(filtered) - 5} more')

## 5. Generate the audio (the slow part)

In [ ]:
import os
from pydub import AudioSegment

if speaker_wav_path:
    uploaded_speaker = files.upload()
    speaker_wav_path = list(uploaded_speaker.keys())[0]

for i, row in enumerate(filtered, 1):
    out_path = row['filename']
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    wav_path = out_path.replace('.mp3', '.wav')
    print(f'[{i}/{len(filtered)}] {out_path}')

    kwargs = {'text': row['text'], 'language': XTTS_LANG_CODE, 'file_path': wav_path}
    if speaker_wav_path:
        kwargs['speaker_wav'] = speaker_wav_path
    else:
        kwargs['speaker'] = tts.speakers[0] if tts.speakers else None

    tts.tts_to_file(**kwargs)
    AudioSegment.from_wav(wav_path).export(out_path, format='mp3')
    os.remove(wav_path)

print(f'\nDone — {len(filtered)} files generated.')

## 6. Zip it and download to your phone
Re-run cells 4–6 with `DIALECT` changed to generate Austria and Switzerland too before downloading, or download after each dialect — either works.

In [ ]:
!zip -r audio_output.zip audio
files.download('audio_output.zip')